# Diabetes Risk Prediction — Backend Notebook

**Goal:** predict whether a patient has diabetes (`Outcome`: 1 = diabetic, 0 = not)
from 8 clinical features, and pick the best model for a Streamlit frontend.

**Data:** the classic Pima Indians Diabetes dataset, 768 rows, in `../data/diabetes.csv`.
**Plan:** clean the data (physiologically impossible zeros -> missing -> median-imputed),
then run **two modelling iterations** on a stratified 80/20 train/test split
(`random_state=42` throughout):
- **Iteration 1 (baseline):** Logistic Regression, Random Forest, SVM, KNN, Gradient Boosting.
- **Iteration 2 (improved):** tune the top 2 Iteration-1 models with 5-fold
  `GridSearchCV` (scoring = F1), including `class_weight='balanced'` where applicable.

**Selection rule:** best model = highest test **F1**, with **Recall** reported explicitly —
for a screening tool, missing a diabetic patient (a false negative) is the costly error.

> Educational demo only — this notebook and its predictions are **not medical advice**
> and must not be used for diagnosis.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
)

# Work whether the notebook is run from notebooks/ or from the repo root.
DATA_DIR = Path("../data") if Path("../data/diabetes.csv").exists() else Path("data")
print("Data dir:", DATA_DIR.resolve())

Data dir: /home/hatch/workspace/your_files/diabetes-risk-app/data


## 1. Load and inspect the raw data

In [2]:
df_raw = pd.read_csv(DATA_DIR / "diabetes.csv")
print("Shape:", df_raw.shape)
print(df_raw.head())
print("\nTarget distribution:")
print(df_raw["Outcome"].value_counts())
print("Diabetic share: %.1f%%" % (100 * df_raw["Outcome"].mean()))
print("\nMissing values in raw file:", int(df_raw.isna().sum().sum()))
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("\nSummary statistics:")
df_raw.describe().round(2)

Shape: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64
Diabetic share: 34.9%

Missing values in raw file: 0
Exact duplicate rows: 0

Summary statistics:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00
mean,3.85,120.89,69.11,20.54,79.80,31.99,0.47,33.24,0.35
std,3.37,31.97,19.36,15.95,115.24,7.88,0.33,11.76,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.30,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,30.50,32.00,0.37,29.00,0.00
75%,6.00,140.25,80.00,32.00,127.25,36.60,0.63,41.00,1.00
max,17.00,199.00,122.00,99.00,846.00,67.10,2.42,81.00,1.00


## 2. Cleaning — physiologically impossible zeros

A living patient cannot have Glucose, BloodPressure, SkinThickness, Insulin or BMI
equal to 0, so zeros in those five columns are recording gaps, not real measurements.
We count them, turn them into `NaN`, and let the modelling pipelines median-impute
(median computed on the **training data only**, inside each pipeline, so no test
information leaks into training).

No rows are dropped: every row still carries usable information in its other columns,
and dropping 652 affected cells' rows would throw away most of the dataset.
There are also **0 duplicate rows**, so nothing is de-duplicated.

In [3]:
ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Impossible-zero counts (raw):")
for c in ZERO_COLS:
    n = int((df_raw[c] == 0).sum())
    print(f"  {c}: {n} zeros ({100 * n / len(df_raw):.1f}% of rows)")

df = df_raw.copy()
df[ZERO_COLS] = df[ZERO_COLS].replace(0, np.nan)
print("\nNaNs after zero -> NaN conversion:", int(df.isna().sum().sum()))
print("\nMedians used for imputation (computed on non-missing values):")
print(df[ZERO_COLS].median().round(2))

Impossible-zero counts (raw):
  Glucose: 5 zeros (0.7% of rows)
  BloodPressure: 35 zeros (4.6% of rows)
  SkinThickness: 227 zeros (29.6% of rows)
  Insulin: 374 zeros (48.7% of rows)
  BMI: 11 zeros (1.4% of rows)

NaNs after zero -> NaN conversion: 652

Medians used for imputation (computed on non-missing values):
Glucose          117.0
BloodPressure     72.0
SkinThickness     29.0
Insulin          125.0
BMI               32.3
dtype: float64


### Saved cleaned file

For the Streamlit app we also save a fully cleaned CSV. Note the difference from the
pipelines: the pipelines above impute from **training data only** (leakage-safe),
while the saved `diabetes_cleaned.csv` is median-imputed on the **full data** so the
app has one complete, ready-to-use file. Its medians are the same values shown above.

In [4]:
df_cleaned = df.copy()
df_cleaned[ZERO_COLS] = df_cleaned[ZERO_COLS].fillna(df_cleaned[ZERO_COLS].median())
df_cleaned.to_csv(DATA_DIR / "diabetes_cleaned.csv", index=False)

chk = pd.read_csv(DATA_DIR / "diabetes_cleaned.csv")
print("Cleaned shape:", chk.shape)
print("NaNs in cleaned file:", int(chk.isna().sum().sum()))
print("Remaining zeros in the five columns:",
      {c: int((chk[c] == 0).sum()) for c in ZERO_COLS})
assert chk.shape[0] == 768 and chk.isna().sum().sum() == 0
print("Cleaned file verified: 768 rows, no NaNs, no impossible zeros.")

Cleaned shape: (768, 9)
NaNs in cleaned file: 0
Remaining zeros in the five columns: {'Glucose': 0, 'BloodPressure': 0, 'SkinThickness': 0, 'Insulin': 0, 'BMI': 0}
Cleaned file verified: 768 rows, no NaNs, no impossible zeros.


## 3. Train/test split

Stratified 80/20 split, `random_state=42`, so the diabetic share (~34.9%) is preserved
in both sets.

In [5]:
FEATURES = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
            "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
X, y = df[FEATURES], df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Test positives: {int(y_test.sum())} / {len(y_test)}")

Train: 614 rows, Test: 154 rows
Test positives: 54 / 154


## 4. Iteration 1 — Baseline models

Every model sits inside an `sklearn` Pipeline: median imputer first, then a scaler
where the model needs one (logistic regression, SVM, KNN), then the classifier.
Tree models skip the scaler (they do not need it). Metrics are computed on the
**test set only**.

In [6]:
def evaluate(model, X_test, y_test):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }

def make_pipeline(model, scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

iter1_models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": make_pipeline(RandomForestClassifier(random_state=42), scale=False),
    "SVM": make_pipeline(SVC(probability=True, random_state=42)),
    "KNN": make_pipeline(KNeighborsClassifier()),
    "Gradient Boosting": make_pipeline(GradientBoostingClassifier(random_state=42), scale=False),
}

iter1_rows = []
for name, model in iter1_models.items():
    model.fit(X_train, y_train)
    row = {"Model": name, **evaluate(model, X_test, y_test)}
    iter1_rows.append(row)

iter1 = pd.DataFrame(iter1_rows).sort_values("F1", ascending=False).reset_index(drop=True)
print("ITERATION 1 — TEST-SET METRICS (sorted by F1)")
iter1.round(4)

ITERATION 1 — TEST-SET METRICS (sorted by F1)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest,0.7662,0.6957,0.5926,0.6400,0.8210
1,KNN,0.7532,0.6600,0.6111,0.6346,0.7899
2,Gradient Boosting,0.7532,0.6818,0.5556,0.6122,0.8309
3,SVM,0.7403,0.6522,0.5556,0.6000,0.7964
4,Logistic Regression,0.7078,0.6000,0.5000,0.5455,0.8130


**Iteration 1 reading:** Random Forest leads on F1, KNN is second. Those two go
forward to tuning in Iteration 2.

## 5. Iteration 2 — Tuning the top 2 models

`GridSearchCV`, 5-fold cross-validation **on the training set only**, scoring = F1.
`class_weight='balanced'` is included in the Random Forest grid because the classes
are imbalanced (500 negative / 268 positive).

In [7]:
rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(random_state=42)),
])
rf_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__class_weight": [None, "balanced"],
}
gs_rf = GridSearchCV(rf_pipe, rf_grid, cv=5, scoring="f1", n_jobs=-1)
gs_rf.fit(X_train, y_train)
print("Best Random Forest params:", gs_rf.best_params_)
print("Best CV F1 (training folds): %.4f" % gs_rf.best_score_)

knn_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier()),
])
knn_grid = {
    "model__n_neighbors": [3, 5, 7, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}
gs_knn = GridSearchCV(knn_pipe, knn_grid, cv=5, scoring="f1", n_jobs=-1)
gs_knn.fit(X_train, y_train)
print("\nBest KNN params:", gs_knn.best_params_)
print("Best CV F1 (training folds): %.4f" % gs_knn.best_score_)

iter2 = pd.DataFrame([
    {"Model": "Random Forest (tuned)", **evaluate(gs_rf.best_estimator_, X_test, y_test)},
    {"Model": "KNN (tuned)", **evaluate(gs_knn.best_estimator_, X_test, y_test)},
]).sort_values("F1", ascending=False).reset_index(drop=True)
print("\nITERATION 2 — TUNED TEST-SET METRICS (sorted by F1)")
iter2.round(4)

Best Random Forest params: {'model__class_weight': 'balanced', 'model__max_depth': 5, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best CV F1 (training folds): 0.6998



Best KNN params: {'model__n_neighbors': 11, 'model__p': 2, 'model__weights': 'distance'}
Best CV F1 (training folds): 0.6356

ITERATION 2 — TUNED TEST-SET METRICS (sorted by F1)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest (tuned),0.7468,0.6056,0.7963,0.6880,0.8185
1,KNN (tuned),0.7338,0.6327,0.5741,0.6019,0.7944


## 6. Did Iteration 2 beat Iteration 1?

In [8]:
best_iter1_f1 = float(iter1.loc[0, "F1"])
best_iter2_f1 = float(iter2.loc[0, "F1"])
best_iter1_rec = float(iter1.loc[0, "Recall"])
best_iter2_rec = float(iter2.loc[0, "Recall"])
print(f"Best Iteration 1 F1: {best_iter1_f1:.4f} ({iter1.loc[0, 'Model']})")
print(f"Best Iteration 2 F1: {best_iter2_f1:.4f} ({iter2.loc[0, 'Model']})")
print(f"F1 improvement: {best_iter2_f1 - best_iter1_f1:+.4f}")
print(f"Recall improvement: {best_iter2_rec - best_iter1_rec:+.4f} "
      f"({best_iter1_rec:.4f} -> {best_iter2_rec:.4f})")
print("Verdict: Iteration 2 DID beat Iteration 1 on F1, driven mainly by a large "
      "Recall gain — the tuned model misses far fewer diabetic patients.")

Best Iteration 1 F1: 0.6400 (Random Forest)
Best Iteration 2 F1: 0.6880 (Random Forest (tuned))
F1 improvement: +0.0480
Recall improvement: +0.2037 (0.5926 -> 0.7963)
Verdict: Iteration 2 DID beat Iteration 1 on F1, driven mainly by a large Recall gain — the tuned model misses far fewer diabetic patients.


## 7. Chosen best model — detailed report

Best by test F1: the **tuned Random Forest**. Confusion matrix layout:
`[[TN, FP], [FN, TP]]` — the bottom-left cell (FN) is the dangerous one for screening,
and tuning cut it sharply.

In [9]:
best_model = gs_rf.best_estimator_
best_pred = best_model.predict(X_test)
best_proba = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Confusion matrix [[TN, FP], [FN, TP]]:\n{cm}")
print(f"True Negatives: {tn}, False Positives: {fp}, "
      f"False Negatives: {fn}, True Positives: {tp}")
print(f"\nMissed diabetic patients (FN): {fn} of {fn + tp} actual positives")
print("\nClassification report:")
print(classification_report(y_test, best_pred,
                            target_names=["Not Diabetic", "Diabetic"]))

importances = pd.Series(
    best_model.named_steps["model"].feature_importances_, index=FEATURES
).sort_values(ascending=False)
print("\nFeature importances (tuned Random Forest):")
print(importances.round(4))

Confusion matrix [[TN, FP], [FN, TP]]:
[[72 28]
 [11 43]]
True Negatives: 72, False Positives: 28, False Negatives: 11, True Positives: 43

Missed diabetic patients (FN): 11 of 54 actual positives

Classification report:
              precision    recall  f1-score   support

Not Diabetic       0.87      0.72      0.79       100
    Diabetic       0.61      0.80      0.69        54

    accuracy                           0.75       154
   macro avg       0.74      0.76      0.74       154
weighted avg       0.78      0.75      0.75       154


Feature importances (tuned Random Forest):
Glucose                     0.3308
BMI                         0.1858
Age                         0.1239
Insulin                     0.1149
DiabetesPedigreeFunction    0.0818
SkinThickness               0.0585
Pregnancies                 0.0571
BloodPressure               0.0473
dtype: float64


## 8. Conclusion

- **Cleaning:** 652 physiologically impossible zeros converted to missing and
  median-imputed; no rows dropped; 0 duplicates found.
- **Iteration 1 winner:** Random Forest (test F1 0.6400).
- **Iteration 2 winner and overall best model:** tuned Random Forest
  (`n_estimators=200, max_depth=5, min_samples_split=5, class_weight='balanced'`)
  with test **F1 0.6880, Recall 0.7963, Accuracy 0.7468, Precision 0.6056,
  ROC-AUC 0.8185** — a +0.0480 F1 gain over Iteration 1, with Recall up +0.2037.
- **Top predictors:** Glucose, BMI and Age.
- The Streamlit app (`../app.py`) retrains exactly this tuned pipeline on the full
  cleaned data at startup, so no binary model file is needed.

> Reminder: educational demo only — not a medical diagnosis. Anyone concerned
> about diabetes should see a healthcare professional.